In [1]:
from upath import UPath
from parkrun_scraper_sdk import ParkrunDataExtractionOrchestrator, Country, Course, CountriesHandler, CoursesHandler, ProcessingConfig, ResultsHandler, EventsHandler#, RunnersHandler



In [2]:
processing_date = "2015-12-24"
country_ids=["3"]
course_ids=[]

# country_ids=[#'3','4','14',
#     '23'
#     #,'30','31','32','42','44','46','54','57','64','65','67','74','82','85','88','97','98'
#              ]
# course_ids=[]

base_path = UPath("/home/nathanielramm/parkrun_data")

config = ProcessingConfig(
    base_path=base_path,
    processing_date=processing_date,
    country_ids=country_ids,
    course_ids=course_ids
)
config.normalize_ids()
config.validate()

event_orchestrator = ParkrunDataExtractionOrchestrator(config=config)

event_orchestrator.update_countries()
event_orchestrator.update_courses()
#TODO: Notify of changes!


courses_in_config_scope = event_orchestrator.get_courses_in_config_scope()
course_ids_in_config_scope = [course.course_id for course in courses_in_config_scope]
# courses_to_process


#TODO: Need tp store last_updated date for each course. Only process if the max date of events is less than the processing date.

# Update events
# for course in courses_to_process:
#     event_orchestrator.events_handler.update_event_history(course=course)


Initialising stored course event ids by course
Initialising stored result event ids by course


In [5]:
import ibis
from ibis import _

dbconn = ibis.duckdb.connect(database="/home/nathanielramm/parkrun_data/parkrun.duckdb")

tbl_results_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/results/country_id=*/course_id=*/*.parquet', union_by_name=True)
tbl_events_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/events/country_id=*/*.parquet', union_by_name=True)
tbl_courses_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/courses/courses.parquet', union_by_name=True)
tbl_countries_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/countries/countries.parquet', union_by_name=True)

all_unprocessed_events = ( tbl_events_raw
        .select(["course_id", "event_id", "event_date"])
        .filter(_.course_id.isin(course_ids_in_config_scope))
        .filter(ibis.date(_.event_date) <= ibis.date(processing_date))
     .left_join(right=tbl_results_raw.select(["course_id", "event_id", "athlete_id"]), predicates=[tbl_events_raw.event_id.cast('string') == tbl_results_raw.event_id.cast('string') 
                                             , tbl_events_raw.course_id.cast('string') == tbl_results_raw.course_id.cast('string')])
     .filter(_.athlete_id.isnull())
     
)

inscope_unprocessed_even_ids = set(all_unprocessed_events.select("course_id").execute()["course_id"].to_list())

# inscope_unprocessed_even_ids


In [6]:
#Update results - will need a faster way to assess what has already been processed!
courses_to_process = [course for course in courses_in_config_scope if course.course_id in inscope_unprocessed_even_ids]

for course in courses_to_process:
    event_orchestrator.results_handler.process_event_results(events_handler=event_orchestrator.events_handler, course=course)

known_event_ids: ['45', '367', '478', '474', '371', '153', '333', '267', '76', '131', '397', '356', '354', '286', '182', '155', '92', '171', '178', '313', '207', '146', '355', '377', '311', '366', '453', '423', '256', '450', '407', '418', '495', '472', '266', '291', '183', '50', '490', '416', '194', '319', '246', '93', '142', '537', '85', '73', '363', '391', '506', '383', '316', '91', '413', '358', '21', '452', '10', '503', '484', '139', '395', '225', '252', '519', '52', '270', '78', '347', '533', '314', '531', '405', '191', '196', '306', '29', '497', '282', '432', '360', '180', '232', '271', '494', '184', '68', '411', '210', '403', '242', '126', '302', '231', '23', '109', '493', '325', '136', '516', '40', '60', '95', '529', '7', '35', '71', '372', '61', '138', '475', '67', '263', '128', '536', '463', '69', '290', '350', '412', '106', '501', '177', '288', '12', '158', '511', '264', '115', '130', '235', '369', '258', '278', '518', '404', '285', '351', '223', '59', '233', '361', '326', '

In [ ]:
orchestrator.countries_handler.get_raw_countries_ids()
# orchestrator.courses_handler.get_raw_courses_by_country_id(country_id="14")


In [ ]:

courses_to_process

In [ ]:
#Update events
for course in courses_to_process:
    orchestrator.events_handler.update_event_history(course=course, )

In [ ]:
import duckdb

# duckdb.sql("SELECT * FROM '/home/nathanielramm/parkrun_data/results/country_id=4/course_id=*/*.parquet'", params={"union_by_name": True})
all_results = duckdb.read_parquet(file_glob="/home/nathanielramm/parkrun_data/results/country_id=*/course_id=*/*.parquet", union_by_name=True)

all_results.s


In [3]:
import ibis
from ibis import _

dbconn = ibis.duckdb.connect(database="/home/nathanielramm/parkrun_data/parkrun.duckdb")

tbl_results_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/results/country_id=*/course_id=*/*.parquet', union_by_name=True)
tbl_events_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/events/country_id=*/*.parquet', union_by_name=True)
tbl_courses_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/courses/courses.parquet', union_by_name=True)
tbl_countries_raw: ibis.Table = dbconn.read_parquet(source_list='/home/nathanielramm/parkrun_data/countries/countries.parquet', union_by_name=True)



In [14]:
#Find events with unprocessed results up to the processing date






,course_id,event_id,event_date,course_id_right,event_id_right,athlete_id
0,553,106,2015-08-22,None,None,None
1,553,101,2015-07-18,None,None,None
2,553,98,2015-06-27,None,None,None
3,553,78,2015-01-17,None,None,None
4,553,62,2014-09-13,None,None,None
...,...,...,...,...,...,...
5408,1153,7,2015-08-08,None,None,None
5409,1342,6,2015-11-21,None,None,None
5410,691,5,2015-01-17,None,None,None
5411,691,1,2014-12-13,None,None,None


In [ ]:
# runners = ( tbl_results_raw
#            .group_by(["athlete_id", "gender"])
#             .aggregate(total_runs=_.gender.count())
#             .select(["athlete_id", "gender"])
#            .group_by(["athlete_id"])
#             .aggregate(total_genders=_.gender.count())
#             .order_by(ibis.desc("total_genders"))
# )

# runners.execute()

tbl_results_raw.group_by(["age_group"]).aggregate(total_runs=_.athlete_id.count()).order_by(ibis.desc("total_runs")).execute()

In [ ]:

def format_time(time_col):

    clean_time = time_col.replace(':', '')
    # Convert to string and pad left with zeros to ensure 6 digits
    padded = clean_time.cast('string').lpad(6, '0')
    
    # Extract hours, minutes, seconds
    hours = padded.substr(0, 2)
    minutes = padded.substr(2, 2)
    seconds = padded.substr(4, 2)
    
    # Combine with colons
    return hours + ':' + minutes + ':' + seconds


def time_to_seconds(time_col):
    # Split on colons and get components

    parts = time_col.split(':')
    hours = parts[0].cast('int64')
    minutes = parts[1].cast('int64')
    seconds = parts[2].cast('int64')
    
    # Convert to total seconds
    # hours * 3600 + minutes * 60 + seconds
    return (hours * 3600) + (minutes * 60) + seconds


v_lkp_course_series = tbl_courses_raw.select(["course_id", "series_id"]).distinct()


v_events_raw = (tbl_events_raw.left_join(v_lkp_course_series, predicates=[tbl_events_raw.course_id == v_lkp_course_series.course_id])
                .select(["course_id","country_id","event_id","event_date", "series_id",
                         "finishers","volunteers" ])
                .mutate(course_id=tbl_events_raw.course_id.cast(target_type=str),
                        country_id=tbl_events_raw.country_id.cast(target_type=str),
                        event_id=tbl_events_raw.event_id.cast(target_type=str),
                        event_id_int=tbl_events_raw.event_id.cast(target_type=int),                        
                        event_date=ibis.date(tbl_events_raw.event_date),
                        series_id=v_lkp_course_series.series_id,

                        finishers=tbl_events_raw.finishers.cast(target_type=int),
                        volunteers=tbl_events_raw.volunteers.cast(target_type=int)
                        )
                )

v_event_winners_raw = (tbl_events_raw.left_join(right=v_lkp_course_series, predicates=[tbl_events_raw.course_id == v_lkp_course_series.course_id])
                       
                .select(["course_id","country_id","event_id","event_date", "series_id", 
                         "male_first_athlete_name","female_first_athlete_name", "male_time", "female_time", "male_athlete_number", "female_athlete_number"])
                .mutate(course_id=tbl_events_raw.course_id.cast(target_type=str),
                        country_id=tbl_events_raw.country_id.cast(target_type=str),
                        event_id=tbl_events_raw.event_id.cast(target_type=str),
                        event_id_int=tbl_events_raw.event_id.cast(target_type=int),                        
                        event_date=ibis.date(tbl_events_raw.event_date),

                        male_athlete_number=tbl_events_raw.male_athlete_number.cast(target_type=str),
                        female_athlete_number=tbl_events_raw.female_athlete_number.cast(target_type=str),

                        male_first_athlete_name=tbl_events_raw.male_first_athlete_name.cast(target_type=str),
                        female_first_athlete_name=tbl_events_raw.female_first_athlete_name.cast(target_type=str),

                        male_time_raw=tbl_events_raw.male_time.cast(target_type=str),
                        female_time_raw=tbl_events_raw.female_time.cast(target_type=str),

                        male_time_corrected = format_time(time_col=tbl_events_raw.male_time),
                        female_time_corrected = format_time(time_col=tbl_events_raw.female_time)
                )
                .mutate(

                        male_time_seconds = time_to_seconds(time_col=_.male_time_corrected),
                        female_time_seconds = time_to_seconds(time_col=_.female_time_corrected)
                )
)

# v_event_winners_raw.execute()




v_event_winners_raw.filter(v_event_winners_raw.series_id == "1", v_event_winners_raw.male_time_seconds.notnull()).order_by(ibis.asc("male_time_seconds")).execute()

# v_event_winners_raw.filter(v_event_winners_raw.series_id == "1").aggregate(by=["country_id", "course_id", "series_id"],  record_male_winner_time=_.male_time_seconds.min()).order_by(ibis.asc("record_male_winner_time")).execute()


                        # male_time_seconds=ibis.cast(ibis.split_part(tbl_events_raw.male_time, ":", 1) * 60 + ibis.split_part(tbl_events_raw.male_time, ":", 2), target_type=int),
                        # female_time_seconds=ibis.cast(ibis.split_part(tbl_events_raw.female_time, ":", 1) * 60 + ibis.split_part(tbl_events_raw.female_time, ":", 2), target_type=int),


In [ ]:
tbl_courses_raw.filter(tbl_courses_raw.course_id == "452").execute()

In [ ]:

v_country_id_max_date = v_events_raw.group_by("country_id").aggregate(max_country_date=_.event_date.max()).order_by(ibis.asc("max_country_date"))
v_course_id_max_date = v_events_raw.group_by(["course_id", "country_id"]).aggregate(max_event_date=_.event_date.max()).order_by(ibis.asc("max_event_date"))
v_course_max_event_date = v_course_id_max_date.left_join(v_country_id_max_date, predicates=[v_course_id_max_date.country_id == v_course_id_max_date.country_id]).mutate(date_diff=v_country_id_max_date.max_country_date - v_course_id_max_date.max_event_date)

v_course_max_event_date.filter(v_course_max_event_date.date_diff > 6).execute()
# course_id_max_date.execute()

In [ ]:
tbl_results_raw.group_by("athlete_id").aggregate("athlete_id").count()

In [ ]:

# orchestrator.events_handler.get_processed_event_ids(course=course)
# orchestrator.events_handler.get_raw_course_event_ids(course=course)

# orchestrator.results_handler.process_event_results(events_handler=orchestrator.events_handler, course=course, event=event)


# orchestrator.results_handler.get_stored_result_event_ids(course=course)


In [ ]:
import requests

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
url = 'https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=IBM&interval=5min&apikey=demo'
r = requests.get(url)
data = r.json()

print(data)